# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task Type: Ranking / Scoring

Explanation:
Our core problem is "Which ones first?" Because an editorial team has limited capacity (e.g., they can only refresh 10–20 articles a week), presenting an un-ordered list of underperforming content is useless. We need to output a continuous priority score for each piece of content, which allows us to sort the entire catalog into a ranked queue from highest expected impact to lowest.

## 2. Target or proxy

The Ultimate Target: The actual traffic or revenue lift generated after an editor refreshes an article.

The Proxy (Observed in Data): Since we cannot observe a future human action (the edit) in historical snapshot data, we use a proxy target: High Impression / Lagging Position. Specifically, we want to predict a continuous value representing "High-Impression Opportunity" (e.g., items with substantial search impressions but an average search position between 6 and 20—meaning they sit at the bottom of Page 1 or on Page 2 of Google search results). A small optimization here yields the highest potential traffic spike.

## 3. Success metric

Metric: Precision@K (specifically Precision@10 or Precision@20)Explanation:Because this is a ranking pipeline designed for human operations, the editors will only look at the very top of the queue (the top $K$ items). If we give them a list of 20 articles to refresh, and 15 of them turn out to be high-leverage opportunities that actually deserved an optimization, our Precision@20 is 75%. We care far more about the precision at the top of the deck than overall model accuracy across 30,000 rows.

## 4. The unit of analysis, as a real dataframe

In this section, you need to execute code to load your starter dataset slice and explicitly define the unit of analysis.

Markdown Cell explanation:

Unit of Analysis: One row = One pseudonymized content item for a specific client, summarizing its performance over a trailing 90-day window.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('content_refresh_anonymized.csv')

df['avg_position'] = df['avg_position'].replace(0, np.nan)

in_striking_distance = df['avg_position'].between(6, 15).astype(int)

log_impressions = np.log1p(df['impressions_90d'])

df['target_priority_score'] = log_impressions * in_striking_distance

print(f"Dataset Shape: {df.shape}")
print("\nUnit of Analysis Display (One row per content item):")
df[['content_id', 'client_id', 'impressions_90d', 'avg_position', 'target_priority_score']].head()

Dataset Shape: (30000, 45)

Unit of Analysis Display (One row per content item):


,content_id,client_id,impressions_90d,avg_position,target_priority_score
0,content_304f48230142,client_f369cb89fc,3803,10.6,8.243808
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.000000
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.000000
3,content_331d6c4de07b,client_19581e27de,11751,6.2,9.371779
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.000000


## 5. Why ML beats a fixed rule here

A plain rule like if impressions > 5000 and position > 7 fails fundamentally because it lacks context and scalability across FlyRank’s 32 completely different clients.

Client Baselines Differ: An article with 1,000 impressions might be a massive breakout flagship piece for a niche boutique Client A, but complete background noise for a massive enterprise Client B. A hardcoded threshold fails to adapt to these scale differences.

High-Dimensional Interaction: A simple rule cannot smoothly balance behavioral metrics (like scroll_rate or engagement_rate) against visibility metrics (ctr, avg_position). If an article has terrible position but a massive scroll rate when people do find it, it's a prime refresh candidate. ML handles these non-linear, messy interactions across 40+ columns simultaneously without a human having to manually write thousands of nested if/else lines.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.